In [13]:
reset()
import numpy as np
import networkx as nx
from numpy import linalg as LA
import random
from fpylll import IntegerMatrix,LLL,GSO
from fpylll.util import gaussian_heuristic

# uses fpylll to return the gram schimdt matrix and the ||bi*||
def gram_schimdt(A, n):
    M = GSO.Mat(A)
    M.update_gso()
    rows = A.nrows
    matrix = np.array([M.get_mu(i, j) for i in range(rows) for j in range(rows)]).reshape(rows, rows)
    orth_norm = np.array([M.get_r(k, k) for k in range(n)])
    return matrix, orth_norm

def mat2fp(A):
    L = IntegerMatrix(A.shape[0], A.shape[1])
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            L[i, j] = int(A[i, j])
    return L

def get_qij(superbasis):
    n = len(superbasis)
    Q = [superbasis[i].dot_product(superbasis[j]) for i in range(n) for j in range(n)]
    Q = matrix(n, n, Q)
    return Q

def gen_ntru_vfk(N,P,q,k):
    H = matrix(N)
    I=identity_matrix(N)
    Zero_Matrix=matrix(N)
    B_1=block_matrix([[I,-k*I]])        
    B_2=block_matrix([[P*I,(q-P*k)*I]])
    M_NTRU=block_matrix([[B_1],[B_2]])
    return M_NTRU

def get_superbasis(basis): # basis is a list
    n = len(basis)
    basis = [vector(x) for x in basis]
    superbasis_vector = -sum(basis[i] for i in [0..n-1])
    return [vector(basis[i]) for i in [0..n-1]] + [vector(superbasis_vector)]

def t_coeffs(dimension, set_c, source):
    set_c.remove(source)
    t = np.zeros(shape=dimension)
    for i in set_c:
        t[i-1] = 1
    return t

def do_mincut_svp(n, Q):
    G = create_weighted_graph_networkx_svp(n, Q) # S depends on the target vector.
    source=0
    sink=n
    # Gsage=Graph(G)
    #for i in range(n+3):
    #    for j in range(i+1,n+3):
    #        print(i,"-->",j,":",Gsage.edge_label(i,j))
    cut_value, partition = nx.minimum_cut(G, source, sink)
    reachable, non_reachable = partition
    cut = reachable if source in reachable else non_reachable
    t_c = t_coeffs(n+1,cut,0)
    return t_c, cut_value

def create_weighted_graph_networkx_svp(dimension, Q):
    """
    source and sink are two positive integers, source = 0 and sink = n+2
    S=(s_i) is given by s_i=-2sum_{j} q_{ij}p_j
    Graph has n+3 nodes: the 1st node is the source = 0, the next n+1 nodes represent each vector 
    of the superbasis and thelast n+3 th node is the sink
    """
    graph = nx.Graph()
    source = 0
    sink = dimension
    graph.add_nodes_from(range(source, sink))

    for i in range(source, sink+1):
        for j in range(source, sink+1):
            # print(i,j,-Q[i][j])
            graph.add_edge(i, j, capacity=-Q[i][j])
    return graph

def is_vfk(Q,N):
    '''
    Q is the Gram-Schimdt matrix of the superbasis
    N is the parameter given when we build ntru_vfk_matrix
    '''
    lst=[]
    x=np.array([Q])
    lst=x[np.nonzero(x)]
    lst=list(lst)
    for i in range(2*N+1):
        if Q[i][i]!=0:
            lst.remove(Q[i][i])
    return all(val <= 0 for val in lst)

def svp_vfk(N,Q,super_basis):
    t, weight = do_mincut_svp(2*N,Q)
    print("t: ", t)
    # print("length from weight: ", np.sqrt(weight))
    vec = np.dot(t, super_basis).astype(int)
    print("length :", np.linalg.norm(vec))
    print("vector=",vec)
    return vec

In [14]:
alpha=10
N,P,q,k=10,6,7*alpha,alpha
M=gen_ntru_vfk(N,P,q,k)
basis = M.rows()
super_basis=get_superbasis(basis)
Q=get_qij(super_basis)
# print("Selling: ", Q)
target_vector = [random.randint(0,1) for m in range(N)]+ [random.randint(-1000,1000) for m in range(N)]
print("Target: ", target_vector)
matrix, orthnorm = gram_schimdt(mat2fp(np.array(basis)) ,2*N)
# matrix should be vector of squared gramshcmit norms
heuristic = np.sqrt(gaussian_heuristic(orthnorm))
print("Heuristic: ", heuristic)

Target:  [0, 0, 0, 1, 0, 1, 1, 0, 1, 0, -537, -651, 854, 26, -195, 986, 26, -478, -248, 800]
Heuristic:  10.045284362775407


In [15]:
from fpylll import IntegerMatrix,LLL
M_NTRU_fplll = IntegerMatrix.from_matrix(basis)
LLL.reduction(M_NTRU_fplll, delta =0.99 )
vector = M_NTRU_fplll[-1]
print(vector)
print(np.linalg.norm(vector))


(0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -10)
10.04987562112089


In [16]:
result = svp_vfk(N,Q,super_basis)

t:  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
length : 10.04987562112089
vector= [  0   0   0   0   0   0   0   0   0   1   0   0   0   0   0   0   0   0
   0 -10]


In [17]:
is_vfk(Q,N)

True